The foundational equations for pricing claims can be found at the bottom of this file


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
import scipy.stats as stats
from matplotlib.ticker import FuncFormatter

from sklearn import svm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.inspection import permutation_importance

from sklearn.model_selection import KFold, GroupKFold, cross_validate, train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from datetime import datetime

import os
import sys

from configparser import ConfigParser
from pathlib import Path

# Sets the working directory so that the key files can be imported
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src" / "insurance_pricing").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

# Make your source package importable
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from Insurance_pricing.data import Find_N_datasets, inspect_dataset, Dates_to_times, missing_length_by_age
from Insurance_pricing.cleaning import clean_types, clean_dates

Risk is defined to be "the potential that actual financial or operational outcomes will differ from the assumptions, models, and estimates used by actuaries."

It can be calcuated using a number of equations however for this project. We will focus primarily on:

**The Collective Risk Model**

$$S = \sum_{i=1}^{N}{X_i}$$
*Total expected losses \(S\), the frequency of claims \(N\), the severity of claims \(X\).*

**General Linear Models \(GLMs\)**


In [3]:
paths = Find_N_datasets(N_files=2, path_loc='processed')
motor = pd.read_csv(paths[0], sep=";")
claims = pd.read_csv(paths[1], sep=";")

motor_data: C:\Users\alloh\OneDrive\Documents\Github\Motor-Insurance-Claims-Prediction-and-Pricing\Data\Processed\Motor_Clean.csv
claims_data: C:\Users\alloh\OneDrive\Documents\Github\Motor-Insurance-Claims-Prediction-and-Pricing\Data\Processed\Claims_Clean.csv


In [ ]:
motor.head()

,ID,Distribution_channel,Seniority,Policies_in_force,Max_policies,Max_products,Lapse,Date_lapse,Payment,Premium,...,Young_Policy_Maker,Contract_Terminated,Is_Motorbike,Is_Van,Is_Passenger_Car,Is_Agricultural,Fuel_Petrol,Fuel_Diesel,Type_fuel_nan,Heavy_vehicle
0,1,0,4,1,2,1,0,NaN,0,222.52,...,0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0
1,1,0,4,1,2,1,0,NaN,0,213.78,...,0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0
2,1,0,4,2,2,1,0,NaN,0,214.84,...,0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0
3,1,0,4,2,2,1,0,NaN,0,216.99,...,0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0
4,2,0,4,2,2,1,0,NaN,1,213.70,...,0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0


In [5]:
motor.columns

Index(['ID', 'Distribution_channel', 'Seniority', 'Policies_in_force',
       'Max_policies', 'Max_products', 'Lapse', 'Date_lapse', 'Payment',
       'Premium', 'Cost_claims_year', 'N_claims_year', 'N_claims_history',
       'R_Claims_history', 'Area', 'Second_driver', 'Year_matriculation',
       'Power', 'Cylinder_capacity', 'Value_vehicle', 'N_doors', 'Length',
       'Weight', 'Date_lapse_missing', 'Fuel_Unknown', 'Length_missing',
       'Policy_Age', 'Licence_Age', 'Policy_Maker_Age', 'Lapse_Age',
       'Time_from_renewal', 'Time_till_renewal', 'Renewed_period',
       'Old_Policy', 'Old_Policy_Maker', 'Young_Policy', 'Young_Policy_Maker',
       'Contract_Terminated', 'Is_Motorbike', 'Is_Van', 'Is_Passenger_Car',
       'Is_Agricultural', 'Fuel_Petrol', 'Fuel_Diesel', 'Type_fuel_nan',
       'Heavy_vehicle'],
      dtype='str')

In [7]:
motor['Policies_in_force'].unique()

array([ 1,  2,  3,  4,  5,  8,  6,  9, 10, 14,  7, 12, 11, 13, 15, 17, 16])

In [ ]:
df_motor = motor[['ID', 'Policies_in_force',]].copy()

df_motor.rename({'Policies_in_force' : 'Frequency'})
df_motor[''] = (

)

# Foundational Equations Used in Actuarial Insurance Pricing

## 1. Claim Frequency and Severity

### Claim Frequency

Measures how often claims occur relative to the number of policies insured.

$$
\lambda = \frac{\text{Number of Claims}}{\text{Number of Policies}}
$$

### Claim Severity

Measures the average monetary cost of an individual claim.

$$
E[X] = \frac{\text{Total Cost of Claims}}{\text{Number of Claims}}
$$

### Expected Claims Cost (Pure Premium)

The expected claims cost per policy is calculated by multiplying claim frequency by claim severity.

$$
\boxed{E[S] = \lambda E[X]}
$$

Where:
- $S$ = Aggregate claims cost.
- $\lambda$ = Expected number of claims per policy.
- $X$ = Cost of an individual claim.
- $E[X]$ = Expected claim severity.

---

## 2. Probability Distributions for Claims

### Poisson Distribution (Claim Frequency)

Models the probability of observing a particular number of claims over a specified period.

$$
\boxed{P(N=n)=\frac{e^{-\lambda}\lambda^n}{n!}}
$$

Where:
- $N$ = Number of claims.
- $n$ = Specific number of claims.
- $\lambda$ = Expected number of claims.
- $e$ = Euler's number.

The probability of receiving zero claims is:

$$
P(N=0)=e^{-\lambda}
$$

### Lognormal Distribution (Claim Severity)

Models positive, right-skewed claim amounts where large claims occur less frequently.

$$
\ln(X)\sim N(\mu,\sigma^2)
$$

Where:
- $X$ = Claim severity.
- $\mu$ = Mean of the logarithm of claim severity.
- $\sigma^2$ = Variance of the logarithm of claim severity.

### Gamma Distribution (Claim Severity)

An alternative distribution for modelling positive, right-skewed claim amounts.

$$
f(x)=\frac{x^{\alpha-1}e^{-x/\theta}}{\Gamma(\alpha)\theta^\alpha}, \quad x>0
$$

Where:
- $\alpha$ = Shape parameter.
- $\theta$ = Scale parameter.
- $\Gamma(\alpha)$ = Gamma function.

---

## 3. Aggregate Claims

### Total Aggregate Claims

Represents the total monetary value of all claims received by an insurer.

$$
\boxed{S=\sum_{i=1}^{N}X_i}
$$

Where:
- $S$ = Total aggregate claims.
- $N$ = Number of claims.
- $X_i$ = Cost of the individual $i$-th claim.

### Expected Aggregate Claims

Assuming claim frequency and severity are independent:

$$
\boxed{E[S]=E[N]E[X]}
$$

For a Poisson frequency model:

$$
E[S]=\lambda E[X]
$$

### Variance of Aggregate Claims

Assuming independent and identically distributed claim severities, and independence between frequency and severity:

$$
\boxed{\operatorname{Var}(S)=E[N]\operatorname{Var}(X)+\operatorname{Var}(N)(E[X])^2}
$$

For a Poisson frequency model:

$$
\boxed{\operatorname{Var}(S)=\lambda E[X^2]}
$$

This measures the uncertainty surrounding the total expected claims cost.

---

## 4. Generalised Linear Models (GLMs)

GLMs allow insurers to estimate how customer characteristics influence expected claim frequency and severity.

### Poisson GLM (Claim Frequency)

$$
\boxed{\log(\lambda_i)=\beta_0+\sum_{j=1}^{p}\beta_jx_{ij}}
$$

Rearranging to calculate expected claim frequency:

$$
\lambda_i=e^{\beta_0+\sum_{j=1}^{p}\beta_jx_{ij}}
$$

Where:
- $\lambda_i$ = Expected claim frequency for customer $i$.
- $\beta_0$ = Intercept.
- $\beta_j$ = Model coefficients.
- $x_{ij}$ = Customer characteristics or predictor variables.
- $p$ = Number of predictor variables.

### Gamma GLM (Claim Severity)

Using a logarithmic link function:

$$
\boxed{\log(E[X_i])=\gamma_0+\sum_{j=1}^{p}\gamma_jx_{ij}}
$$

Where:
- $E[X_i]$ = Expected claim severity for customer $i$.
- $\gamma_0$ = Intercept.
- $\gamma_j$ = Model coefficients.
- $x_{ij}$ = Customer characteristics or predictor variables.

### Customer-Specific Pure Premium

Combining the frequency and severity models:

$$
\boxed{P_i=\lambda_i E[X_i]}
$$

Where:
- $P_i$ = Expected claims cost for customer $i$.
- $\lambda_i$ = Predicted claim frequency.
- $E[X_i]$ = Predicted claim severity.

---

## 5. Deductibles and Insurance Limits

Insurance policies often include excesses (deductibles) and maximum payment limits.

### Insurer's Claim Payment

$$
\boxed{Y=\min(\max(X-d,0),L)}
$$

Where:
- $Y$ = Amount paid by the insurer.
- $X$ = Total claim amount.
- $d$ = Policy excess or deductible.
- $L$ = Maximum insurer payment limit.

### Expected Insurer Claim Payment

$$
\boxed{E[Y]=\int_d^{d+L}P(X>x)\,dx}
$$

This calculates the expected amount paid by the insurer after accounting for the deductible and policy limit.

---

## 6. Final Insurance Premium

The pure premium only covers expected claims. Insurers must also account for operating expenses and their target profit margin.

### Gross Insurance Premium

$$
\boxed{P=\frac{EL+F}{1-v-m}}
$$

Where:
- $P$ = Gross insurance premium.
- $EL$ = Expected claims cost (pure premium).
- $F$ = Fixed expenses per policy.
- $v$ = Variable expenses as a proportion of premium.
- $m$ = Target profit margin as a proportion of premium.

### Example

Suppose an insurer estimates:

| Component | Value |
|---|---:|
| Expected claims cost ($EL$) | £480 |
| Fixed expenses ($F$) | £60 |
| Variable expenses ($v$) | 20% |
| Target profit margin ($m$) | 5% |

The gross premium is:

$$
P=\frac{480+60}{1-0.20-0.05}
$$

$$
\boxed{P=£720}
$$

---

## 7. Overall Actuarial Pricing Process

The general process combines the above equations:

1. **Historical data:** Collect policyholder characteristics, previous claims and claim amounts.
2. **Frequency modelling:** Estimate the expected number of claims using distributions such as Poisson or Negative Binomial.
3. **Severity modelling:** Estimate the expected cost of claims using distributions such as Gamma or Lognormal.
4. **Expected claims cost:** Combine frequency and severity to calculate the pure premium.
5. **Policy adjustments:** Account for deductibles, limits, inflation and reinsurance.
6. **Final premium:** Include expenses, risk allowances and the insurer's target profit.

The central equation connecting the process is:

$$
\boxed{\text{Expected Claims Cost}=\text{Frequency}\times\text{Severity}}
$$

---

## 8. Additional Actuarial Concepts Worth Learning

| Concept | Application |
|---|---|
| Expected value | Estimating average future claims costs. |
| Variance | Quantifying uncertainty in claims. |
| Poisson distribution | Modelling claim frequency. |
| Negative Binomial distribution | Modelling overdispersed claim frequency. |
| Gamma distribution | Modelling claim severity. |
| Lognormal distribution | Modelling right-skewed claim severity. |
| Compound distributions | Modelling total aggregate losses. |
| Generalised Linear Models | Predicting customer-specific claim frequency and severity. |
| Maximum Likelihood Estimation | Estimating statistical model parameters. |
| Chain Ladder Method | Estimating outstanding insurance claims (reserving). |

**Important distinction:** Pricing estimates the expected cost of future claims to determine premiums. Reserving estimates the outstanding cost of claims that have already occurred but may not yet have been fully settled.